# 03 - Feature Engineering

Derive combustion risk metrics using `src/features.py`.

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Load the cleaned BASS-II dataset
df = pd.read_csv('../data/processed/bass2_cleaned.csv')

# 1. One-Hot Encode Categorical Variables
categorical_cols = ['Fuel Sample Material', 'Flow restrictor']
existing_cats = [c for c in categorical_cols if c in df.columns]
df_encoded = pd.get_dummies(df, columns=existing_cats, drop_first=True)

# 2. Derive Physics-Based Interaction Metric: O2 to CO2 Conversion Ratio
df_encoded['O2_CO2_Ratio'] = df_encoded['O2_Delta'] / (df_encoded['CO2_Delta'] + 1e-5)

# 3. Filter Numerical Features for Machine Learning Models
feature_cols = [
    c for c in df_encoded.columns 
    if c.endswith('_Delta') or 'Fuel Sample Material' in c or 'Flow restrictor' in c or c == 'O2_CO2_Ratio'
]

X = df_encoded[feature_cols]
y = df_encoded['High_Risk_Combustion']

# 4. Standardize Features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Save processed matrices for model training
X_scaled.to_csv('../data/processed/X_features.csv', index=False)
y.to_csv('../data/processed/y_target.csv', index=False)

print("--- Feature Engineering Complete ---")
print(f"Feature Matrix Shape: {X_scaled.shape}")
print("Engineered Features List:")
for col in X_scaled.columns:
    print(f" - {col}")

## Save the feature table

Add a `hazard_label` column (1 = hazardous, 0 = safe/contained) based on known experiment outcomes/metadata before modeling.

In [ ]:
# feature_table.to_csv('../data/processed/feature_table.csv', index=False)